# Data Prep for HMM-LAI

This notebook runs through the data preparation steps to create the reference and admixed test files from raw 1000 Genomes Phase 3 chr21 data.

## Expected Inputs

Files should be housed in: `~/1000genomes:`
- `ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz`
- `ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz.tbi`
- `igsr_samples.tsv`

In [4]:
#if chr21 files have not been downloaded to ./1000genomes folder
from pathlib import Path

base = Path("./1000genomes").expanduser()
base.mkdir(exist_ok=True)

vcf_path = base / "ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz"
tbi_path = base / "ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz.tbi"

# chr21 VCF
if not vcf_path.exists():
    !wget -c https://hgdownload.soe.ucsc.edu/gbdb/hg19/1000Genomes/phase3/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz -P {base}
# index
if not tbi_path.exists():
    !wget -c https://hgdownload.soe.ucsc.edu/gbdb/hg19/1000Genomes/phase3/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz.tbi -P {base}

print("Files ready:")
print(vcf_path)
print(tbi_path)

zsh:1: command not found: wget
zsh:1: command not found: wget
Files ready:
1000genomes/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz
1000genomes/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz.tbi


## Notebook Outputs

Outputs a `./1000genomes/<size>_subset_data_prep/` folder with the following files
- combined all samples VCF 
- reference VCF (using AFR, EUR, EAS samples)
- admixed test VCF (using AMR samples)
- reference panel TSV (mapping each reference sample to AFR, EUR, or EAS)
- sample list txts and metadata files

In [1]:
from pathlib import Path
import os
import subprocess
import pandas as pd

In [5]:
#setting base and output paths
base = Path("./1000genomes").expanduser()

vcf_path = base / "ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz"
samples_path = base / "igsr_samples.tsv"

#small subet (45 ref, 10 test)
#medium subset (150 ref, 30 test)
out_dir = base / "small_subset_data_prep"
out_dir.mkdir(exist_ok=True)

print(vcf_path)
print(samples_path)
print(out_dir)

1000genomes/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz
1000genomes/igsr_samples.tsv
1000genomes/small_subset_data_prep


In [3]:
#inspect sample df
samples_df = pd.read_csv(samples_path, sep="\t")
samples_df = samples_df[samples_df["Data collections"].fillna("").str.contains("Phase 3",case=False)]
samples_df.head()

,Sample name,Sex,Biosample ID,Population code,Population name,Superpopulation code,Superpopulation name,Population elastic ID,Data collections
0,HG00271,male,SAME123417,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."
1,HG00276,female,SAME123424,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."
2,HG00288,female,SAME1839246,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."
3,HG00290,male,SAME1839057,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."
5,HG00308,male,SAME124161,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."


In [4]:
#keep only samples that are in the chr21 phase 3 VCF
!bcftools query -l {vcf_path} > {out_dir}/vcf_samples.txt
vcf_samples = set(pd.read_csv(out_dir/"vcf_samples.txt", header=None)[0])
samples_df = samples_df[samples_df["Sample name"].isin(vcf_samples)].copy()
print(len(samples_df))

2504


In [5]:
#columns of interest
sample_col = "Sample name"
superpop_col = "Superpopulation code"
pop_col = "Population code"

In [8]:
#subsetting df

#========
#small subset
#15 samples for each reference group (45 total)
#10 samples for test group 

#medium subset
#50 samples for each reference group (150 total)
#30 samples for test group

n_ref_per_group = 15
n_test_amr = 10

#========
ref_groups = ["AFR", "EUR", "EAS"]
test_group = "AMR"

ref_parts = []
for g in ref_groups:
    part = samples_df[samples_df[superpop_col] == g].head(n_ref_per_group).copy()
    part["panel"] = g
    ref_parts.append(part)

ref_df = pd.concat(ref_parts, ignore_index=True)
test_df = samples_df[samples_df[superpop_col] == test_group].head(n_test_amr).copy()

# write sample lists
ref_list = out_dir / "ref_samples.txt"
test_list = out_dir / "admixed_amr_samples.txt"
all_list = out_dir / "all_selected_samples.txt"

ref_df[[sample_col]].to_csv(ref_list, index=False, header=False)
test_df[[sample_col]].to_csv(test_list, index=False, header=False)

all_df = pd.concat([ref_df[[sample_col]], test_df[[sample_col]]], ignore_index=True).drop_duplicates()
all_df.to_csv(all_list, index=False, header=False)

print("sample lists:")
print(ref_list)
print(test_list)
print(all_list)
print('\n')

# write metadata
ref_meta = out_dir / "ref_metadata.tsv"
test_meta = out_dir / "test_metadata.tsv"

ref_df.to_csv(ref_meta, sep="\t", index=False)
test_df.to_csv(test_meta, sep="\t", index=False)

print("metadata:")
print(ref_meta)
print(test_meta)
print('\n')

# write reference panel
ref_panel = out_dir / "ref_panel.tsv"
ref_df[[sample_col, "panel"]].to_csv(ref_panel, sep="\t", header=False, index=False)

print("reference panel:")
print(ref_panel)

sample lists:
1000genomes/small_subset_data_prep/ref_samples.txt
1000genomes/small_subset_data_prep/admixed_amr_samples.txt
1000genomes/small_subset_data_prep/all_selected_samples.txt


metadata:
1000genomes/small_subset_data_prep/ref_metadata.tsv
1000genomes/small_subset_data_prep/test_metadata.tsv


reference panel:
1000genomes/small_subset_data_prep/ref_panel.tsv


In [9]:
#start with a combined VCF file that is filtered according:
# -biallelic SNPS only
# -missingness <=1%
# -minor allele count >=5
# procedure follows FLARE paper

combined_filtered_vcf = out_dir / "all_selected.filtered.chr21.vcf.gz"

!bcftools view \
    -S {all_list} \
    -m2 -M2 \
    -v snps \
    -i 'F_MISSING<=0.01 && MAC>=5' \
    -Oz \
    -o {combined_filtered_vcf} \
    {vcf_path}

!bcftools index -t {combined_filtered_vcf}

In [ ]:
#split the combined VCF(filtered) into the reference and test VCFs
ref_vcf = out_dir / "reference_AFR_EUR_EAS.filtered.chr21.vcf.gz"
test_vcf = out_dir / "admixed_AMR.filtered.chr21.vcf.gz"

#reference set
!bcftools view \
    -S {ref_list} \
    -Oz \
    -o {ref_vcf} \
    {combined_filtered_vcf}

!bcftools index -t {ref_vcf}

#test set
!bcftools view \
    -S {test_list} \
    -Oz \
    -o {test_vcf} \
    {combined_filtered_vcf}

!bcftools index -t {test_vcf}

In [13]:
# subset VCFs so they are small enough for our tool to run
subset_ref_vcf = out_dir / "reference_AFR_EUR_EAS.filtered.subset.chr21.vcf"
subset_test_vcf = out_dir / "admixed_AMR.filtered.subset.chr21.vcf"
!gzcat {ref_vcf} | grep "^#" > {subset_ref_vcf}
!gzcat {ref_vcf} | grep -v "^#" | awk 'NR % 100 == 0' >> {subset_ref_vcf}
!bgzip {subset_ref_vcf}
!bcftools index -t {subset_ref_vcf}'.gz'

!gzcat {test_vcf} | grep "^#" > {subset_test_vcf}
!gzcat {test_vcf} | grep -v "^#" | awk 'NR % 100 == 0' >> {subset_test_vcf}
!bgzip {subset_test_vcf}
!bcftools index -t {subset_test_vcf}'.gz'

In [15]:
#confirm sample counts
print("reference sample count:")
!bcftools query -l {subset_ref_vcf}'.gz' | wc -l

print("\ntest sample count:")
!bcftools query -l {subset_test_vcf}'.gz' | wc -l

print("\nreference filtered variant count:")
!bcftools index -n {subset_ref_vcf}'.gz'

print("\ntest filtered variant count:")
!bcftools index -n {subset_test_vcf}'.gz'


reference sample count:
      45

test sample count:
      10

reference filtered variant count:
3860

test filtered variant count:
3860
